In [3]:
%%time
import numpy as np
import h5py
import re
import csv
import pandas as pd

##### IMPORT ARCHS4 H5PY DATA #####  

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/"
file_path = "refining/data_slash_results/"
filename = root + file_path + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'

with h5py.File(filename, 'r') as f:
    gene_names = f['meta/genes/all_gene_names'][:]
    gene_ids = f['meta/genes/gene_id'][:]
arch_gene_names = [x.decode('UTF-8') for x in gene_names]
arch_gene_ids = [x.decode('UTF-8') for x in gene_ids]

##### IMPORT G-PROFILER MATCHED GENES DATA ##### 

file_path = "/refining/gene_length_correction/match_genes/"
filename = root + file_path + "gProfiler_archs4_geneSymbol_geneIDs.csv"
matched_genes_df = pd.read_csv(filename)

CPU times: user 168 ms, sys: 38.8 ms, total: 207 ms
Wall time: 252 ms


In [5]:
print("ARCHS4 Gene IDS:", arch_gene_ids[0:3])
print("ARCHS4 Gene Size:", len(arch_gene_ids))
print()
print("gProfiler Matched Genes Size:",len(matched_genes_df))
print("gProfiler Matched Genes Size:",matched_genes_df['description'].tolist()[0:3])

ARCHS4 Gene IDS: ['ENSG00000121410', 'ENSG00000268895', 'ENSG00000148584']
ARCHS4 Gene Size: 57447

gProfiler Matched Genes Size: 63022
gProfiler Matched Genes Size: ['alpha-1-B glycoprotein [Source:HGNC Symbol;Acc:HGNC:5]', 'A1BG antisense RNA 1 [Source:HGNC Symbol;Acc:HGNC:37133]', 'APOBEC1 complementation factor [Source:HGNC Symbol;Acc:HGNC:24086]']


In [7]:
#**** GET GENE DESCRIPTION LIST ****# 
df_gene_description = matched_genes_df['description'].tolist()

# Extract the string before "[Source:]"
matched_genes_df['description'] = matched_genes_df['description'].str.replace(r'\s*\[Source:.*\]', '', regex=True)
print("gProfiler Matched Genes Size:",matched_genes_df['description'].tolist()[0:3])


gProfiler Matched Genes Size: ['alpha-1-B glycoprotein', 'A1BG antisense RNA 1', 'APOBEC1 complementation factor']


In [8]:
#**** GET MATCHED GENE THROUGH ID, GET DESCRIPTION LIST ****# 

# assuming that arch_gene_ids and matched_genes_df['converted_alias'] are lists
df_gene_ids = matched_genes_df['converted_alias'].tolist()

descriptions = []
for gene_id in arch_gene_ids:
    if gene_id in df_gene_ids:
        descriptions.append(matched_genes_df.loc[matched_genes_df['converted_alias'] == gene_id, 'description'].tolist()[0])

print("DF Description Size:", len(descriptions))
print(descriptions[0:3])

DF Description Size: 57447
['alpha-1-B glycoprotein', 'A1BG antisense RNA 1', 'APOBEC1 complementation factor']


In [21]:
#**** SAVE & SEE DESCRIPTION LIST ****# 
with open('descriptions.txt', 'w') as f:
    for line in descriptions:
        f.write(f"{line}\n")

In [27]:
#**** RESOLVED ERROR IN SAVING H5PY  ****# 
# TypeError: No conversion path for dtype: dtype('<U169')

descriptions_np = np.array(descriptions, dtype='S169')
print(descriptions_np[0:3])
descriptions_np = [x.decode('UTF-8') for x in descriptions_np]
print(descriptions_np[0:3])

[b'alpha-1-B glycoprotein' b'A1BG antisense RNA 1'
 b'APOBEC1 complementation factor']
['alpha-1-B glycoprotein', 'A1BG antisense RNA 1', 'APOBEC1 complementation factor']


In [28]:
#---- SAVED IN H5PY  ----# 
root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/"
file_path = "refining/data_slash_results/"
filename = root + file_path + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'

file = h5py.File(filename, "a")

# create the pathway "meta/genes/" if it does not exist
if "meta" not in file:
    file.create_group("meta")
if "genes" not in file["meta"]:
    print("Error")
    
group = file['meta/genes']
print(group.keys())

# write the list to the dataset in the pathway "meta/genes/"
file["meta/genes/gene_description"] = descriptions_np

group = file['meta/genes']
print(group.keys())

# close the h5py file
file.close()


<KeysViewHDF5 ['all_gene_names', 'gene_id', 'gene_length']>
<KeysViewHDF5 ['all_gene_names', 'gene_description', 'gene_id', 'gene_length']>
